# EXP-040 | Multi-seed 10 Seeds × 3 Models = 30모델 (Colab GPU 버전)

EXP-038(5 seeds, OOF 0.74090)의 검증된 아키텍처를 10 seeds로 확장.

| 근거 | 수치 |
|---|---|
| 1→5 seed 개선 (EXP-032→EXP-038) | OOF +0.00019 |
| 5→10 seed 예상 개선 (수확 체감) | OOF +0.00009~0.00010 |
| 목표 OOF | ≥ 0.74098 (1등 0.74246 돌파 조건) |

**GPU 변경사항**: CatBoost `task_type='GPU'` / XGBoost `tree_method='gpu_hist'`

**1등 조건**: OOF+0.00148(갭) = 제출점수. 0.74098+0.00148 = **0.74246** → 1등 동등

| 기준선 | EXP-038 OOF 0.74090 / EXP-039 OOF 0.74091 |

In [ ]:
!pip install lightgbm catboost xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

warnings.filterwarnings('ignore')

# ── 경로 설정: train.csv, test.csv, sample_submission.csv 을 /content/ 에 업로드하세요
DATA_DIR = Path('/content')
OUT_DIR  = Path('/content')

TARGET   = '임신 성공 여부'
N_FOLDS  = 5
SEEDS    = [42, 0, 123, 2024, 777, 1, 99, 314, 2023, 500]
EXP_NO   = 40
AUTHOR   = '조여진'

print(f'Seeds({len(SEEDS)}): {SEEDS}')
print(f'총 {len(SEEDS)*3*N_FOLDS}회 학습  (3모델×{len(SEEDS)}시드×{N_FOLDS}폴드)')

## 0. 전처리 (src/preprocessing.py 인라인)

In [ ]:
CONST_COLS = ['불임 원인 - 여성 요인', '난자 채취 경과일']
RECODE_ZERO_COLS = ['착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부']

COUNT_ORDER = ['0회', '1회', '2회', '3회', '4회', '5회', '6회 이상']
AGE_ORDER = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세', '알 수 없음']
DONOR_AGE_ORDER = ['만20세 이하', '만21-25세', '만26-30세', '만31-35세', '만36-40세', '만41-45세', '알 수 없음']

ORDINAL_COLS = {
    '시술 당시 나이': AGE_ORDER, '난자 기증자 나이': DONOR_AGE_ORDER, '정자 기증자 나이': DONOR_AGE_ORDER,
    '총 시술 횟수': COUNT_ORDER, '클리닉 내 총 시술 횟수': COUNT_ORDER,
    'IVF 시술 횟수': COUNT_ORDER, 'DI 시술 횟수': COUNT_ORDER,
    '총 임신 횟수': COUNT_ORDER, 'IVF 임신 횟수': COUNT_ORDER, 'DI 임신 횟수': COUNT_ORDER,
    '총 출산 횟수': COUNT_ORDER, 'IVF 출산 횟수': COUNT_ORDER, 'DI 출산 횟수': COUNT_ORDER,
}
LABEL_COLS = ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처']
EMBRYO_COL = '배아 생성 주요 이유'
EMBRYO_REASONS = ['기증용', '난자 저장용', '배아 저장용', '연구용', '현재 시술용']

def preprocess(train_df, test_df):
    tr = train_df.drop(columns=['ID', TARGET] + CONST_COLS, errors='ignore').copy()
    te = test_df.drop(columns=['ID'] + CONST_COLS, errors='ignore').copy()
    for col in RECODE_ZERO_COLS:
        if col in tr.columns:
            tr[col] = tr[col].fillna(0).astype(int)
            te[col] = te[col].fillna(0).astype(int)
    for col, order in ORDINAL_COLS.items():
        if col in tr.columns:
            mapping = {v: i for i, v in enumerate(order)}
            tr[col] = tr[col].map(mapping)
            te[col] = te[col].map(mapping)
    if EMBRYO_COL in tr.columns:
        for reason in EMBRYO_REASONS:
            col_name = f'배아이유_{reason}'
            tr[col_name] = tr[EMBRYO_COL].fillna('').str.contains(reason).astype(int)
            te[col_name] = te[EMBRYO_COL].fillna('').str.contains(reason).astype(int)
        tr.drop(columns=[EMBRYO_COL], inplace=True)
        te.drop(columns=[EMBRYO_COL], inplace=True)
    for col in LABEL_COLS:
        if col in tr.columns:
            le = LabelEncoder()
            le.fit(tr[col].astype(str))
            tr[col] = le.transform(tr[col].astype(str))
            classes_map = {c: i for i, c in enumerate(le.classes_)}
            te[col] = te[col].astype(str).map(classes_map).fillna(len(le.classes_)).astype(int)
    return tr, te

print('전처리 함수 정의 완료')

## 1. 데이터 로드 & 피처 준비

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv', encoding='cp949')
test  = pd.read_csv(DATA_DIR / 'test.csv',  encoding='cp949')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='cp949')
print(f'train: {train.shape}  /  test: {test.shape}')

def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features_v1(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

def add_derived_features_v2(df):
    df = add_derived_features_v1(df)
    eps = 1e-6
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

TE_COLS   = ['시술 시기 코드','시술 유형','특정 시술 유형','배란 유도 유형',
             '난자 출처','정자 출처','시술 당시 나이','총 시술 횟수','총 임신 횟수']
ITE_PAIRS = [('시술 당시 나이','시술 유형'),('시술 당시 나이','특정 시술 유형'),
             ('시술 당시 나이','난자 출처'),('시술 당시 나이','배란 유도 유형'),
             ('시술 유형','총 시술 횟수'),('난자 출처','시술 유형')]
TE_ALPHA, ITE_ALPHA = 10, 20

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_base_tr, X_base_te = preprocess(train_fe, test_fe)

X_v1_train = add_derived_features_v1(X_base_tr)
X_v1_test  = add_derived_features_v1(X_base_te)
X_v2_train = add_derived_features_v2(X_base_tr)
X_v2_test  = add_derived_features_v2(X_base_te)

TE_COLS   = [c for c in TE_COLS   if c in X_v2_train.columns]
ITE_PAIRS = [(c1,c2) for c1,c2 in ITE_PAIRS
             if c1 in X_v2_train.columns and c2 in X_v2_train.columns]

y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'FE-v1: {X_v1_train.shape[1]}개  /  FE-v2: {X_v2_train.shape[1]}개')
print(f'TE: {len(TE_COLS)}개  /  ITE: {len(ITE_PAIRS)}쌍')

## 2. 모델 파라미터 (GPU 적용)

In [ ]:
LGB_PARAMS_BASE = dict(
    objective='binary', metric='auc', verbosity=-1,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824,
    num_leaves=266, max_depth=5, min_child_samples=166,
    feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091,
    lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442,
    min_gain_to_split=0.11418176854933762,
)
# CatBoost GPU: bootstrap_type='Bernoulli' 필수, colsample_bylevel(rsm) GPU 미지원으로 제거
CAT_PARAMS_BASE = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', thread_count=-1, verbose=False,
    early_stopping_rounds=100,
    task_type='GPU', bootstrap_type='Bernoulli',
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483,
)
# XGBoost 2.0+: gpu_hist 제거됨 → tree_method='hist' + device='cuda'
XGB_PARAMS_BASE = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist', device='cuda',
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)

def rank_avg_normalize(arrays):
    ranks = np.stack([rankdata(a) for a in arrays], axis=1)
    avg = ranks.mean(axis=1)
    return (avg - avg.min()) / (avg.max() - avg.min())

print('파라미터 설정 완료 (CatBoost GPU+Bernoulli / XGBoost hist+cuda)')
print('모델 구성: LGB(FE-v1) + CAT(FE-v2+TE) + XGB(FE-v2+TE+ITE)  ×  10 seeds')

## 3. Multi-seed 10 Seeds 학습

In [ ]:
n_train = len(X_v1_train)
n_test  = len(X_v1_test)

all_oof_lgb  = []
all_oof_cat  = []
all_oof_xgb  = []
all_test_lgb = []
all_test_cat = []
all_test_xgb = []

for s_idx, seed in enumerate(SEEDS, 1):
    print(f'\n{"="*60}')
    print(f'Seed {seed}  ({s_idx}/{len(SEEDS)})  [{"기존" if seed in [42,0,123,2024,777] else "신규"}]')
    print(f'{"="*60}')

    lgb_p = {**LGB_PARAMS_BASE, 'seed': seed}
    cat_p = {**CAT_PARAMS_BASE, 'random_seed': seed}
    xgb_p = {**XGB_PARAMS_BASE, 'random_state': seed}
    skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_lgb  = np.zeros(n_train)
    oof_cat  = np.zeros(n_train)
    oof_xgb  = np.zeros(n_train)
    test_lgb = np.zeros(n_test)
    test_cat = np.zeros(n_test)
    test_xgb = np.zeros(n_test)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v1_train, y_train), 1):
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        global_mean = y_tr.mean()

        # ── LGB: FE-v1 ──────────────────────────────────────
        X_tr_lgb  = X_v1_train.iloc[tr_idx]
        X_val_lgb = X_v1_train.iloc[val_idx]
        ds_tr  = lgb.Dataset(X_tr_lgb, label=y_tr)
        ds_val = lgb.Dataset(X_val_lgb, label=y_val, reference=ds_tr)
        m = lgb.train(lgb_p, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                      callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(-1)])
        oof_lgb[val_idx] = m.predict(X_val_lgb)
        test_lgb        += m.predict(X_v1_test) / N_FOLDS

        # ── TE/ITE 공통 계산 (fold 내부) ────────────────────
        X_tr_v2  = X_v2_train.iloc[tr_idx].copy()
        X_val_v2 = X_v2_train.iloc[val_idx].copy()
        X_te_v2  = X_v2_test.copy()
        for col in TE_COLS:
            grp = y_tr.groupby(X_tr_v2[col])
            sm  = (grp.count() * grp.mean() + TE_ALPHA * global_mean) / (grp.count() + TE_ALPHA)
            tc  = f'{col}_te'
            X_tr_v2[tc]  = X_tr_v2[col].map(sm).fillna(global_mean)
            X_val_v2[tc] = X_val_v2[col].map(sm).fillna(global_mean)
            X_te_v2[tc]  = X_te_v2[col].map(sm).fillna(global_mean)

        X_tr_xgb  = X_tr_v2.copy()
        X_val_xgb = X_val_v2.copy()
        X_te_xgb  = X_te_v2.copy()
        for col1, col2 in ITE_PAIRS:
            key_tr  = X_tr_xgb[col1].astype(str)  + '_' + X_tr_xgb[col2].astype(str)
            key_val = X_val_xgb[col1].astype(str) + '_' + X_val_xgb[col2].astype(str)
            key_te  = X_te_xgb[col1].astype(str)  + '_' + X_te_xgb[col2].astype(str)
            grp = y_tr.groupby(key_tr)
            sm  = (grp.count() * grp.mean() + ITE_ALPHA * global_mean) / (grp.count() + ITE_ALPHA)
            ic  = f'{col1}_{col2}_ite'
            X_tr_xgb[ic]  = key_tr.map(sm).fillna(global_mean)
            X_val_xgb[ic] = key_val.map(sm).fillna(global_mean)
            X_te_xgb[ic]  = key_te.map(sm).fillna(global_mean)

        # ── CAT: FE-v2+TE (GPU) ─────────────────────────────
        m = CatBoostClassifier(**cat_p)
        m.fit(X_tr_v2, y_tr, eval_set=Pool(X_val_v2, y_val), use_best_model=True)
        oof_cat[val_idx] = m.predict_proba(X_val_v2)[:, 1]
        test_cat        += m.predict_proba(X_te_v2)[:, 1] / N_FOLDS

        # ── XGB: FE-v2+TE+ITE (GPU) ─────────────────────────
        m = xgb.XGBClassifier(**xgb_p)
        m.fit(X_tr_xgb, y_tr, eval_set=[(X_val_xgb, y_val)], verbose=False)
        oof_xgb[val_idx] = m.predict_proba(X_val_xgb)[:, 1]
        test_xgb        += m.predict_proba(X_te_xgb)[:, 1] / N_FOLDS

        fold_lgb = roc_auc_score(y_val, oof_lgb[val_idx])
        fold_cat = roc_auc_score(y_val, oof_cat[val_idx])
        fold_xgb = roc_auc_score(y_val, oof_xgb[val_idx])
        print(f'  Fold {fold}  LGB={fold_lgb:.5f}  CAT={fold_cat:.5f}  XGB={fold_xgb:.5f}')

    seed_oof = rank_avg_normalize([oof_lgb, oof_cat, oof_xgb])
    seed_auc = roc_auc_score(y_train, seed_oof)
    print(f'  → Seed {seed} OOF (Rank Avg LGB+CAT+XGB): {seed_auc:.5f}')

    all_oof_lgb.append(oof_lgb)
    all_oof_cat.append(oof_cat)
    all_oof_xgb.append(oof_xgb)
    all_test_lgb.append(test_lgb)
    all_test_cat.append(test_cat)
    all_test_xgb.append(test_xgb)

print('\n학습 완료')

## 4. Rank Average — OOF & Test

In [ ]:
all_oofs  = all_oof_lgb  + all_oof_cat  + all_oof_xgb
all_tests = all_test_lgb + all_test_cat + all_test_xgb

oof_final  = rank_avg_normalize(all_oofs)
test_final = rank_avg_normalize(all_tests)

auc_final = roc_auc_score(y_train, oof_final)

BASELINE_032 = 0.74071
BASELINE_038 = 0.74090
TARGET_1ST   = 0.74098
GAP          = 0.00148

print(f'[EXP-040] OOF AUC (30모델 Rank Avg): {auc_final:.5f}')
print(f'  vs EXP-032: {auc_final - BASELINE_032:+.5f}')
print(f'  vs EXP-038: {auc_final - BASELINE_038:+.5f}')
print(f'  1등 기준(≥0.74098): {"✓ 달성" if auc_final >= TARGET_1ST else f"✗ 미달 ({TARGET_1ST - auc_final:.5f} 부족)"}')
print(f'  예상 제출 점수: {auc_final + GAP:.5f}  (OOF + {GAP})')

oof_5s = rank_avg_normalize(all_oofs[:5] + all_oofs[10:15] + all_oofs[20:25])
auc_5s = roc_auc_score(y_train, oof_5s)
print(f'\n[비교]  5-seed(기존): {auc_5s:.5f}  →  10-seed(신규): {auc_final:.5f}  ({auc_final-auc_5s:+.5f})')

print('\n시드별 OOF AUC:')
for i, seed in enumerate(SEEDS):
    sa = roc_auc_score(y_train, rank_avg_normalize([all_oof_lgb[i], all_oof_cat[i], all_oof_xgb[i]]))
    tag = '기존' if seed in [42,0,123,2024,777] else '신규'
    print(f'  Seed {seed:>4d} [{tag}]: {sa:.5f}')

## 5. Submission 저장

> 리더보드 기록은 로컬에서 수동으로 추가하세요.

In [ ]:
auc_str   = f'{auc_final:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub_out   = sub.copy()
sub_out['probability'] = test_final
sub_out.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장 완료: {OUT_DIR / out_fname}')
print(f'probability 범위: [{test_final.min():.4f}, {test_final.max():.4f}]')
print(f'\n최종 OOF: {auc_final:.5f}  /  예상 제출: {auc_final + GAP:.5f}')